# arena-interface walkthrough

A tour of the library against the **simulated controller**, so it runs without hardware.
To use a real arena, change `SIMULATE` to `False` (the controller is found automatically)
or pass a port name to `Device.open()`.

In [ ]:
from arena_interface import Device, FrameRecord, GrayLevel, Pattern, TrialMode

SIMULATE = True
dev = Device.open(simulate=SIMULATE)
print(dev.firmware_version, "-", dev.arena, dev.arena.pixel_shape)

## The pattern library on the SD card

In [ ]:
for entry in dev.patterns:
    info = dev.patterns.info(entry.pattern_id)
    print(
        entry.pattern_id, entry.name, info.frame_count, "frames", info.gray_level.name
    )

## Make a pattern from pixel arrays

Images are `dev.arena.pixel_shape` pixels (`rows * 20` by `cols * 20`); row 0 is the top of the arena.
Plain lists work; numpy arrays work too if numpy is installed.

In [ ]:
h, w = dev.arena.pixel_shape
frames = [
    [[15 if (x - 10 * f) % w < 20 else 0 for x in range(w)] for _ in range(h)]
    for f in range(20)
]
bar = Pattern.from_frames(frames, gray_level=GrayLevel.GS16, arena=dev.arena)
dev.stop()
pid = dev.patterns.upload("notebook_bar.pat", bar, overwrite=True)
bar, pid

## Run a trial and log the frames the arena actually showed

In [ ]:
dev.telemetry.drain()
dev.start_trial(pid, TrialMode.OPEN_LOOP, frame_rate=20, duration=1.0)
shown = [
    (e.time, e.record.index)
    for e in dev.telemetry.events(duration=1.2)
    if isinstance(e.record, FrameRecord)
]
print(len(shown), "frame changes; first few:", shown[:3])

## Sample the analog inputs

In [ ]:
samples = list(dev.sample(dev.io.read_analog_in, rate_hz=100, duration=0.5))
print(len(samples), "samples; last:", samples[-1])

## Clean up

Closing blanks the arena (ALL_OFF) and releases the port.

In [ ]:
dev.patterns.delete(pid)
dev.close()